In [11]:
import os
import torch
import numpy as np
from utils import compute_mae, compute_crps
from dataset import process_dataset
from scaler import VitalScaler


In [12]:
#1. Load and Process Data
train_df, test_df = process_dataset(test_mode=False) 

#Normalize Data and Save Scaler
print("loading datasets...")
scaler = VitalScaler()
scaler.fit(train_df)

loading datasets...


In [15]:
out_path = '../../outputs/Diffusion'

targets = ['HR', 'RESP', 'SpO2']

maes = {}
crpss = {}
for pred_type in targets:
    maes[pred_type] = []
    crpss[pred_type] = []

for batch_name in os.listdir(out_path):
    batch_path = f'{out_path}/{batch_name}'

    encoder_target = scaler.inverse(np.load(f'{batch_path}/encoder_target.npy'))
    decoder_target = scaler.inverse(np.load(f'{batch_path}/decoder_target.npy'))
    samples = np.load(f'{batch_path}/samples.npy')
    median_pred = np.median(samples, axis=0)

    crps = compute_crps(samples, decoder_target, targets)
    mae = compute_mae(median_pred, decoder_target, targets)

    for pred_type in targets:
        maes[pred_type].append(mae[pred_type].item())
        crpss[pred_type].append(crps[pred_type].item())

for pred_type in targets:
    print(f'{pred_type} -- CRPS - Mean: {np.mean(crpss[pred_type])} || Median: {np.median(crpss[pred_type])}')
    print(f'{pred_type} -- MAE - Mean: {np.mean(maes[pred_type])} || Median: {np.median(maes[pred_type])}')

HR -- CRPS - Mean: 1.8596918135171845 || Median: 1.7743641138076782
HR -- MAE - Mean: 2.300258016586304 || Median: 2.178420305252075
RESP -- CRPS - Mean: 1.4924247274796167 || Median: 1.420628309249878
RESP -- MAE - Mean: 1.8695906365201587 || Median: 1.7830785512924194
SpO2 -- CRPS - Mean: 0.6782565283456019 || Median: 0.6063413619995117
SpO2 -- MAE - Mean: 0.8400177973721709 || Median: 0.7574381530284882
